Script om grafiek met lijnen van ZO, NW en NO mee te plotten in 1

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from toolsos.huisstijl.colors import get_os_colors
from graphs import simple_barh, simple_line, multiple_line, grouped_bar
import pickle
from itertools import cycle

In [3]:
df_onbewerkt = pd.read_csv('../../04 tabellen/data_meting2_jun_26_alle_data.csv')
df_onbewerkt.head(1)

,indicator_sd,measure,bron,value,temporal_date,spatial_code,spatial_name,spatial_type,tweedeling_def,thema_zuidoost_label,...,thema_nw_kleur,thema_nw_label,mpzo,aanpak_noord,samen_nw,kernindicator_zo,kernindicator_noord,kernindicator_nw,spatial_date,besch_jaren
0,Kunst en cultuur vergroot de leefwereld van in...,kcgrlfw_p,stadsdeelenquête ZO,33.3,2023,TA,Amstel III/Bullewijk,wijken,zittende bewoner,SD8 Kunst en cultuur,...,NaN,NaN,True,False,False,False,NaN,NaN,20220324,2023|2025


In [4]:
# df_onbewerkt.loc[df_onbewerkt["kernindicator_noord"] == 1][
#     ["thema_noord_eenmeting", "indicator_sd", "measure"]
# ].drop_duplicates().to_excel("kernindicatoren_noord.xlsx")


In [5]:
df_onbewerkt.loc[(df_onbewerkt['measure']=='vkparkeren_r') & (df_onbewerkt['spatial_type']=='gemeente')]['temporal_date'].value_counts()

temporal_date
2019    1
2021    1
2023    1
2025    1
Name: count, dtype: int64

In [6]:
def bewerk_dataframe(df_onbewerkt):
    df = df_onbewerkt.loc[
        (df_onbewerkt["spatial_type"].isin(["stadsdelen", "gemeente"])) &
        (df_onbewerkt["tweedeling_def"] == "totaal")
    ].copy()

    df["temporal_date"] = pd.to_datetime(df["temporal_date"], format="%Y")
    df["temporal_date"] = df["temporal_date"].dt.year.astype(str)

    #df = df.sort_values(["spatial_name", "measure", "temporal_date"])

    return df


df = bewerk_dataframe(df_onbewerkt)

In [7]:
kernindicatoren_zo = df.loc[df['kernindicator_zo']==1]['measure'].unique()
kernindicatoren_nw = df.loc[df['kernindicator_nw']==1]['measure'].unique()
kernindicatoren_no = df.loc[df['kernindicator_noord']==1]['measure'].unique()

# kleur mapping aanmaken per indicator
indicatoren = df['measure'].unique()
kleuren = get_os_colors(type='discreet', kleur='discreet (1-9)', aantal='9')  # Lijst met kleuren
kleur_mapping = {indicatoren: kleur for indicatoren, kleur in zip(indicatoren, cycle(kleuren))}

In [8]:
# selecteer indicatoren met 2025 data

data_2025 = df.loc[df['temporal_date'] == '2025']['measure'].unique().tolist()


In [9]:
#df[['indicator_sd','measure', '']].value_counts().to_excel('lijst_indicatoren.xlsx')

#### Multiple Line - vergelijk ZO, NW en NO

In [10]:
def multiple_line_perc_figuur(df, indicator_list, list_stadsdelen, output_folder):
    for indicator in indicator_list:
        
        list_stadsdelen.append('Amsterdam')
        df_plot = df.loc[(df['measure'] == indicator) & 
        (df['spatial_name'].isin(list_stadsdelen))]


        if df['value'].notna().any():

            kleur = kleur_mapping.get(indicator, '#004699')  # Zoek de kleur op in de mapping

            # Controleer of '_p' of '_r' in de indicatornaam staat en pas y-as daarop aan
            if '_p' in indicator:
                y_min = 0
                y_max = 100 if df_plot['value'].max() > 20 else 25 if df_plot['value'].max() > 10 else 15
                is_percentage = True
            elif '_r' in indicator:
                y_min = 1
                y_max = 10
                is_percentage = False
            else:
                # Gebruik standaardwaarden als geen '_p' of '_r' aanwezig is
                y_min = None
                y_max = None
                is_percentage = False 
                                    
            multiple_line(
                df=df_plot,
                x_col='temporal_date',
                y_col='value',
                main_area='Amsterdam', 
                y_min= y_min,
                y_max= y_max,
                color_func=kleur,  # Gebruik de kleur uit de mapping
                is_percentage=is_percentage,
                output_path=f'../../05 reports/{output_folder}/{indicator}.png'
                )

# Verwerk indicatoren
multiple_line_perc_figuur(df, indicatoren, ['Zuidoost', 'Nieuw-West','Noord'], 'Alle stadsdelen')

# Verwerk kernindicatoren zo
#multiple_line_perc_figuur(df, kernindicatoren_zo, ['Zuidoost'], 'Zuidoost/kernindicatoren')
# multiple_line_perc_figuur(df, kernindicatoren_zo, ['Nieuw-West'], 'Nieuw-West/kernindicatoren')
multiple_line_perc_figuur(df, kernindicatoren_no, ['Noord'], 'Noord/kernindicatoren')

In [11]:
# def make_vertical_bar_graph(df, indicator_list, list_stadsdelen, output_folder):
#     for indicator in indicator_list:
        
#         #if indicator in data_2025: # moet 2025 data van zijn

#             #list_stadsdelen.append('Amsterdam')
#             df_plot = df.loc[(df['measure'] == indicator) & (df['spatial_name'].isin(list_stadsdelen))]
#             df['value'] = df['value'].round(1)
#             if df['value'].notna().any() and '_r' in indicator:
            
#                 kleur = kleur_mapping.get(indicator, '#004699') # Zoek de kleur op in de mapping
#                 print(indicator)

#                 simple_barh(
#                     df=df_plot,
#                     x_col='temporal_date',
#                     y_col='value',
#                     y_min=0,
#                     y_max=10,
#                     color_func=kleur,
#                     output_path=f'../../05 reports/{output_folder}/{indicator}.png'
#                     )

# make_vertical_bar_graph(df, indicatoren, ['Zuidoost'], 'Zuidoost/kernindicatoren')

# # Verwerk kernindicatoren zo
# make_vertical_bar_graph(df, kernindicatoren_zo, ['Zuidoost'], 'Zuidoost/kernindicatoren')
# # make_vertical_bar_graph(df, kernindicatoren_zo, ['Nieuw-West'], 'Nieuw-West/kernindicatoren')
# # make_vertical_bar_graph(df, kernindicatoren_zo, ['Noord'], 'Noord/kernindicatoren')

In [14]:

kleurmap_grouped_bar = {
    "Amsterdam": "#009dec", # lichtblauw
    "Nieuw-West": "#d48fb9",
    "Noord": "#ff9100", #6cbd74",
    "Zuidoost":  "#004699" # donkerblauw
    }

def make_grouped_bar(df, indicator_list, list_stadsdelen, output_folder):
    for indicator in indicator_list:
        
        df_plot = df.loc[(df['measure'] == indicator) & (df['spatial_name'].isin(list_stadsdelen))]
        df['value'] = df['value'].round(1)
        if df['value'].notna().any() and '_r' in indicator:
        
            color_sd = kleurmap_grouped_bar.get(list_stadsdelen[0]) # Zoek de kleur op in de mapping
            color_ams = kleurmap_grouped_bar.get(list_stadsdelen[1]) # Zoek de kleur op in de mapping

            print(indicator)

            grouped_bar(
                df=df_plot,
                x_col='temporal_date',
                y_col='value',
                y_min=0,
                y_max=10,
                color_sd = color_sd,
                color_ams = color_ams,
                output_path=f'../../05 reports/{output_folder}/{indicator}.png'
                )


print('alle indicatoren met 2025 data:')
#make_grouped_bar(df, indicatoren, ['Zuidoost', 'Amsterdam'], 'Zuidoost/indicatoren grouped bars')

# Verwerk kernindicatoren zo
print('\nkerndindicatoren met 2025 data:')
make_grouped_bar(df, kernindicatoren_zo, ['Zuidoost', 'Amsterdam'], 'Zuidoost/kernindicatoren grouped bars')
# make_vertical_bar_graph(df, kernindicatoren_zo, ['Nieuw-West'], 'Nieuw-West/kernindicatoren')
make_grouped_bar(df, kernindicatoren_no, ['Noord', 'Amsterdam'], 'Noord/kernindicatoren grouped bars')

alle indicatoren met 2025 data:

kerndindicatoren met 2025 data:
lomganggroepenb_r
oraanbodspelen_r
vkparkeren_r
lbetrokken_r
lbuurt_r
bhwinkelaanbod_r
wwoningcorp_r
vveiligavond_r
bhhorecaaanbod_r
lomganggroepenb_r
wwoning_r
lbetrokken_r
lbuurt_r
oronderhoudstraten_r
wonderhoudwoning_r


In [37]:
df.to_excel('../../05 reports/Alle stadsdelen/alle_data.xlsx')